In [2]:
import os
import json
import time
import re
import random
from datetime import datetime
from typing import List, Dict, Any, Generator
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pinecone import Pinecone, ServerlessSpec
import fitz  # PyMuPDF
import docx2txt  # Word extraction
import serpapi
import langgraph
import langchain
import serpapi
import wikipedia
from groq import Groq
from langchain_groq import ChatGroq
from tavily import TavilyClient

In [3]:
# ---------------- LOAD ENV VARIABLES ----------------
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
secret_api_key = os.getenv("SERPAPI_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME", "studybuddy")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT", "us-east1-gcp")


In [29]:

if not PINECONE_API_KEY:
    raise ValueError("❌ PINECONE_API_KEY not found!")

#---------------- CONFIGURE GEMINI & PINECONE ----------------
#Initialize Gemini client with latest google-genai SDK
client = genai.Client(api_key=GEMINI_API_KEY)

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create or connect to Pinecone index
existing_indexes = [i.name for i in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_ENVIRONMENT)
    )
index = pc.Index(INDEX_NAME)


In [47]:
from langchain_core.tools import tool
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for information about a topic."""
    try:
        # Search for the query
        search_results = wikipedia.search(query)
        if not search_results:
            return f"No Wikipedia articles found for '{query}'"
        
        # Get the first result
        page = wikipedia.page(search_results[0])
        
        # Return summary
        return f"📚 Wikipedia: {page.title}\n\n{page.summary[:600]}...\n\n🔗 {page.url}"
        
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple pages found for '{query}'. Options: {', '.join(e.options[:5])}"
    except wikipedia.exceptions.PageError:
        return f"No page found for '{query}'"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

In [49]:
@tool 
def web_search(query: str) -> str:
    """Performs a web search using Tavily and returns the top result."""
    client = TavilyClient(TAVILY_API_KEY)
    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=5,
    )
    return response

In [50]:
tools = [ web_search,search_wikipedia]

In [51]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    api_key=GROQ_API_KEY
)


llm_with_tools = llm.bind_tools(tools)

In [52]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# 1. State definition
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Agent Node (Invokes LLM with bound tools)
def call_model(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 3. Tool Node (Executes tools when LLM returns tool_calls)
tool_node = ToolNode(tools)


In [53]:
from langgraph.graph import END

def route_after_model(state: State):
    last_message = state["messages"][-1]
    
    # If the LLM requested a tool call, route to 'tools' node
    if last_message.tool_calls:
        return "tools"
    
    # If no tool calls were requested, stop and return to the user
    return END


In [54]:
from langgraph.graph import StateGraph, START

workflow = StateGraph(State)

# Add our processing units
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Connect flow logic
workflow.add_edge(START, "agent")

# Add conditional routing after the agent node finishes
workflow.add_conditional_edges(
    "agent",
    route_after_model,
    {
        "tools": "tools",  # routes to 'tools' node if route_after_model returns 'tools'
        END: END          # routes to END if route_after_model returns END
    }
)

# Crucial: route back to the agent after tool execution so it can read tool outputs!
workflow.add_edge("tools", "agent")

# Compile into an executable application
app = workflow.compile()


In [59]:
# Scenario A: Triggers 'get_user_balance' tool automatically
input_state = {"messages": [("user", "Use Wikipedia to tell me about Abraham Lincoln and also search the web for today's news on Bitcoin and the crypto market")]}
for chunk in app.stream(input_state):
    print(chunk)



{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bgqxwaxp3', 'function': {'arguments': '{"query":"Abraham Lincoln"}', 'name': 'search_wikipedia'}, 'type': 'function'}, {'id': 'q8w1qv16x', 'function': {'arguments': '{"query":"Bitcoin and crypto market news today"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 771, 'total_tokens': 834, 'completion_time': 0.142321219, 'completion_tokens_details': None, 'prompt_time': 0.038905099, 'prompt_tokens_details': None, 'queue_time': 0.00392319, 'total_time': 0.181226318}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019edbc1-1b65-7ca1-8c4e-2eed31dcb881-0', tool_calls=[{'name': 'search_wikipedia', 'args': {'query': 'Abraham Lincoln'}, 'id': 'bgqxwaxp3', 'type': 'too

In [2]:
%pip install llama_index

%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip uninstall  sentence-transformers torch transformers

^C
Note: you may need to restart the kernel to use updated packages.
